# import

In [ ]:
import os
from pathlib import Path
import joblib
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score
)
import argparse

import plotly.express as px

# funciones extra

In [ ]:
def graficar_3d(X, color):
    fig = px.scatter_3d(
        X,
        x='wind_speed_ms',
        y='wave_period_s',
        z='wave_height_m',
        color=color,
        opacity=0.5
    )
    fig.update_traces(marker_size=3)
    fig.update_layout(
        scene=dict(
            aspectmode='cube'
        )
    )

    return fig

In [ ]:
def graficar_2d(X, color):
    fig = px.scatter(
        X,
        x='datetime',
        y='wave_height_m',
        color=color,
        opacity=0.5
    )
    fig.update_traces(marker_size=3)

    return fig

# configuraciones

In [ ]:
ambiente = 'project'

EXTREME_THRESHOLD = 0.95
EVERY_N_YEARS = 10
PORCENTAJE_SAMPLE_DATA_MONTH = 1
RANDOM_SEED = 0
TEST_SIZE = 0.2

N_CLUSTERS = 6
EXTREME_FEATURES = ['wave_energy', 'wave_power_kW_m']
MODEL_FEATURES = [
    'wind_speed_ms', 
    'wave_energy', 'wave_height_m', 'wave_period_s', 'wave_power_kW_m'
]
SCALED_MODEL_FEATURES = [f'{feature}_scaled' for feature in MODEL_FEATURES]

if 'DATABRICKS_RUNTIME_VERSION' in os.environ:
    model_path = f'/Volumes/cor_{ambiente}/ml/models/wave_classifier/wave_classifier{{}}.pkl'
else:
    base_path = Path.cwd().parent
    model_path = f'{base_path}/wave_classifier/wave_classifier{{}}.pkl'


# proceso separado por celdas

In [ ]:
X_train, X_test = train_test_split(df, test_size=TEST_SIZE, random_state=RANDOM_SEED)

In [ ]:
threshold_energia = X_train["wave_energy"].quantile(EXTREME_THRESHOLD)
threshold_potencia = X_train["wave_power_kW_m"].quantile(EXTREME_THRESHOLD)

In [ ]:
def es_extremo(X):
    return (
        (X["wave_energy"] >= threshold_energia) |
        (X["wave_power_kW_m"] >= threshold_potencia)
    )

In [ ]:
scaler = StandardScaler()
scaler.fit(
    X_train.loc[~es_extremo(X_train), MODEL_FEATURES]
)

In [ ]:
def procesar(X):
    x = X.copy()
    x["es_extremo"] = es_extremo(x)
    x_extremo = x[x['es_extremo']].reset_index(drop=True).drop(columns=['es_extremo'])
    x_no_extremo = x[~x['es_extremo']].reset_index(drop=True).drop(columns=['es_extremo'])

    temp_scaled = scaler.transform(x_no_extremo[MODEL_FEATURES])
    df_temp_scaled = pd.DataFrame(temp_scaled, columns=SCALED_MODEL_FEATURES)

    x_no_extremo = pd.concat([x_no_extremo.reset_index(drop=True), df_temp_scaled], axis=1)
    
    return x_no_extremo, x_extremo

In [ ]:
X_train_no_extremo, X_train_extremo = procesar(X_train)
X_test_no_extremo, X_test_extremo = procesar(X_test)

In [ ]:
gmm = GaussianMixture(
    n_components=N_CLUSTERS,
    covariance_type="full",
    random_state=RANDOM_SEED
)

In [ ]:
gmm.fit(X_train_no_extremo[SCALED_MODEL_FEATURES])

In [ ]:
X_train_no_extremo["gmm_cluster"] = gmm.predict(X_train_no_extremo[SCALED_MODEL_FEATURES])
X_train_no_extremo["gmm_cluster_probability"] = gmm.predict_proba(X_train_no_extremo[SCALED_MODEL_FEATURES]).max(axis=1)

In [ ]:
cluster_summary = (
    X_train_no_extremo.groupby("gmm_cluster")[EXTREME_FEATURES]
    .mean()
    .sort_values(EXTREME_FEATURES)
)
cluster_order = {
    old_cluster: f'{new_cluster + 1}'
    for new_cluster, old_cluster in enumerate(cluster_summary.index)
}

In [ ]:
X_test_no_extremo["gmm_cluster"] = gmm.predict(X_test_no_extremo[SCALED_MODEL_FEATURES])
X_test_no_extremo["gmm_cluster_probability"] = gmm.predict_proba(X_test_no_extremo[SCALED_MODEL_FEATURES]).max(axis=1)

In [ ]:
q_5 = X_test_no_extremo.groupby('gmm_cluster')['gmm_cluster_probability'].quantile(0.05)
q_10 = X_test_no_extremo.groupby('gmm_cluster')['gmm_cluster_probability'].quantile(0.10)
q_15 = X_test_no_extremo.groupby('gmm_cluster')['gmm_cluster_probability'].quantile(0.15)

In [ ]:
if not (
    all(X_test_no_extremo.groupby('gmm_cluster')['gmm_cluster_probability'].quantile(0.05)>=0.7) and
    all(X_test_no_extremo.groupby('gmm_cluster')['gmm_cluster_probability'].quantile(0.10)>=0.8) and
    all(X_test_no_extremo.groupby('gmm_cluster')['gmm_cluster_probability'].quantile(0.15)>=0.9) 
):
    print(f'El modelo GMM no es lo suficientemente bueno, saltando...')
else:
        print(f'Modelo GMM cumple con los criterios')

print(f'Quantiles: 0.05: {q_5}, 0.10: {q_10}, 0.15: {q_15}')

In [ ]:
X_test_no_extremo = X_test_no_extremo.assign(gmm_cluster=lambda df: df["gmm_cluster"].map(cluster_order))
X_test_extremo = X_test_extremo.assign(gmm_cluster='7')

X_train_no_extremo = X_train_no_extremo.assign(gmm_cluster=lambda df: df["gmm_cluster"].map(cluster_order))
X_train_extremo = X_train_extremo.assign(gmm_cluster='7')

In [ ]:
X_train_classified = pd.concat(
    [
        X_train_no_extremo[['coast_name', 'datetime'] + MODEL_FEATURES + ['gmm_cluster']],
        X_train_extremo[['coast_name', 'datetime'] + MODEL_FEATURES + ['gmm_cluster']]
    ],
    ignore_index=True
)

X_test_classified = pd.concat(
    [
        X_test_no_extremo[['coast_name', 'datetime'] + MODEL_FEATURES + ['gmm_cluster']],
        X_test_extremo[['coast_name', 'datetime'] + MODEL_FEATURES + ['gmm_cluster']]
    ],
    ignore_index=True
)

# proceso en 1 funcion

In [ ]:
def proceso(df):
    def es_extremo(X):
        return (
            (X["wave_energy"] >= threshold_energia) &
            (X["wave_power_kW_m"] >= threshold_potencia)
        )
    def procesar(X):
        x = X.copy()
        x["es_extremo"] = es_extremo(x)
        x_extremo = x[x['es_extremo']].reset_index(drop=True).drop(columns=['es_extremo'])
        x_no_extremo = x[~x['es_extremo']].reset_index(drop=True).drop(columns=['es_extremo'])

        temp_scaled = scaler.transform(x_no_extremo[MODEL_FEATURES])
        df_temp_scaled = pd.DataFrame(temp_scaled, columns=SCALED_MODEL_FEATURES)

        x_no_extremo = pd.concat([x_no_extremo.reset_index(drop=True), df_temp_scaled], axis=1)
        
        return x_no_extremo, x_extremo
    
    X_train, X_test = train_test_split(df, test_size=TEST_SIZE, random_state=RANDOM_SEED)

    threshold_energia = X_train["wave_energy"].quantile(EXTREME_THRESHOLD)
    threshold_potencia = X_train["wave_power_kW_m"].quantile(EXTREME_THRESHOLD)

    scaler = StandardScaler()
    scaler.fit(
        X_train.loc[~es_extremo(X_train), MODEL_FEATURES]
    )

    X_train_no_extremo, X_train_extremo = procesar(X_train)
    X_test_no_extremo, X_test_extremo = procesar(X_test)

    gmm = GaussianMixture(
        n_components=N_CLUSTERS,
        covariance_type="full",
        random_state=RANDOM_SEED
    )

    gmm.fit(X_train_no_extremo[SCALED_MODEL_FEATURES])

    X_train_no_extremo["gmm_cluster"] = gmm.predict(X_train_no_extremo[SCALED_MODEL_FEATURES])
    X_train_no_extremo["gmm_cluster_probability"] = gmm.predict_proba(X_train_no_extremo[SCALED_MODEL_FEATURES]).max(axis=1)

    cluster_summary = (
        X_train_no_extremo.groupby("gmm_cluster")[EXTREME_FEATURES]
        .mean()
        .sort_values(EXTREME_FEATURES)
    )
    cluster_order = {
        old_cluster: f'{new_cluster + 1}'
        for new_cluster, old_cluster in enumerate(cluster_summary.index)
    }

    X_test_no_extremo["gmm_cluster"] = gmm.predict(X_test_no_extremo[SCALED_MODEL_FEATURES])
    X_test_no_extremo["gmm_cluster_probability"] = gmm.predict_proba(X_test_no_extremo[SCALED_MODEL_FEATURES]).max(axis=1)

    q_5 = X_test_no_extremo.groupby('gmm_cluster')['gmm_cluster_probability'].quantile(0.05)
    q_10 = X_test_no_extremo.groupby('gmm_cluster')['gmm_cluster_probability'].quantile(0.10)
    q_15 = X_test_no_extremo.groupby('gmm_cluster')['gmm_cluster_probability'].quantile(0.15)
    if not (
        all(X_test_no_extremo.groupby('gmm_cluster')['gmm_cluster_probability'].quantile(0.05)>=0.7) and
        all(X_test_no_extremo.groupby('gmm_cluster')['gmm_cluster_probability'].quantile(0.10)>=0.8) and
        all(X_test_no_extremo.groupby('gmm_cluster')['gmm_cluster_probability'].quantile(0.15)>=0.9) 
    ):
        print(f'El modelo GMM no es lo suficientemente bueno, saltando...')
    else:
            print(f'Modelo GMM cumple con los criterios')

    print(f'Quantiles: 0.05: {q_5}, 0.10: {q_10}, 0.15: {q_15}')

    X_test_no_extremo = X_test_no_extremo.assign(gmm_cluster=lambda df: df["gmm_cluster"].map(cluster_order))
    X_test_extremo = X_test_extremo.assign(gmm_cluster='7')

    X_train_no_extremo = X_train_no_extremo.assign(gmm_cluster=lambda df: df["gmm_cluster"].map(cluster_order))
    X_train_extremo = X_train_extremo.assign(gmm_cluster='7')

    X_train_classified = pd.concat(
        [
            X_train_no_extremo[['coast_name', 'datetime'] + MODEL_FEATURES + ['gmm_cluster']],
            X_train_extremo[['coast_name', 'datetime'] + MODEL_FEATURES + ['gmm_cluster']]
        ],
        ignore_index=True
    )

    X_test_classified = pd.concat(
        [
            X_test_no_extremo[['coast_name', 'datetime'] + MODEL_FEATURES + ['gmm_cluster']],
            X_test_extremo[['coast_name', 'datetime'] + MODEL_FEATURES + ['gmm_cluster']]
        ],
        ignore_index=True
    )

    return X_train_classified, X_test_classified

# obtener datos

In [ ]:
data = (
        spark.sql(
            f"""
                SELECT coast_name, datetime, {', '.join(set(MODEL_FEATURES + EXTREME_FEATURES))},
                CONCAT(coast_name, '_', DATE_FORMAT(datetime, 'yyyyMM')) AS coast_year_month
                FROM cor_{ambiente}.silver.swell_metrics
                WHERE YEAR(datetime) % {EVERY_N_YEARS} = 0
            """
        )
    )

In [ ]:
coast_year_month_dict = {row.coast_year_month: PORCENTAJE_SAMPLE_DATA_MONTH for row in data.select('coast_year_month').distinct().collect()}

df = (
    data
    .sampleBy('coast_year_month', fractions=coast_year_month_dict, seed=RANDOM_SEED)
    .drop('coast_year_month')
).toPandas()

# prueba todos los datos

In [ ]:
X_train_classified, X_test_classified = proceso(df)

In [ ]:
matamoros = X_train_classified[X_train_classified['coast_name']=='Matamoros']

In [ ]:
#agrupar por mes y ver la distribución de clusters
cluster_month = matamoros.groupby([matamoros['datetime'].dt.month, 'gmm_cluster']).size().unstack(fill_value=0)

In [ ]:
px.line(cluster_month).show()

# proceso solo matamoros

In [ ]:
df_2 = df[df['coast_name']=='Matamoros'].reset_index(drop=True)

In [ ]:
X_train_classified, X_test_classified = proceso(df_2)

In [ ]:
matamoros = X_train_classified[X_train_classified['coast_name']=='Matamoros']

In [ ]:
cluster_month = matamoros.groupby([matamoros['datetime'].dt.month, 'gmm_cluster']).size().unstack(fill_value=0)

In [ ]:
px.line(cluster_month).show()

In [27]:
import os
from pathlib import Path
import joblib
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score
)
import argparse

# Argumentos

ambiente = 'project'
print(ambiente)

# Parametros
EXTREME_THRESHOLD = 0.95
EVERY_N_YEARS = 3
PORCENTAJE_SAMPLE_DATA_MONTH = 1
RANDOM_SEED = 0
TEST_SIZE = 0.2

N_CLUSTERS = 6
EXTREME_FEATURES = ['wave_height_m', 'wave_period_s']
MODEL_FEATURES = [
    'wind_speed_ms', 
    'wave_energy', 'wave_height_m', 'wave_period_s', 'wave_power_kW_m', 'wave_energy', 'wave_power_kW_m'
]
SCALED_MODEL_FEATURES = [f'{feature}_scaled' for feature in MODEL_FEATURES]

N_ESTIMATORS = 500
MAX_DEPTH = None
MIN_SAMPLES_LEAF = 10
CLASS_WEIGHT = "balanced"
N_JOBS = -1

if 'DATABRICKS_RUNTIME_VERSION' in os.environ:
    model_path = f'/Volumes/cor_{ambiente}/ml/models/wave_classifier/wave_classifier.pkl'
else:
    base_path = Path.cwd().parent
    model_path = f'{base_path}/wave_classifier/wave_classifier.pkl'

project


In [28]:
def clasificar_gmm(df):
    def es_extremo(X):
        return (
            (X["wave_energy"] >= threshold_energia) &
            (X["wave_power_kW_m"] >= threshold_potencia)
        )
    def procesar(X):
        x = X.copy()
        x["es_extremo"] = es_extremo(x)
        x_extremo = x[x['es_extremo']].reset_index(drop=True).drop(columns=['es_extremo'])
        x_no_extremo = x[~x['es_extremo']].reset_index(drop=True).drop(columns=['es_extremo'])

        temp_scaled = scaler.transform(x_no_extremo[MODEL_FEATURES])
        df_temp_scaled = pd.DataFrame(temp_scaled, columns=SCALED_MODEL_FEATURES)

        x_no_extremo = pd.concat([x_no_extremo.reset_index(drop=True), df_temp_scaled], axis=1)
        
        return x_no_extremo, x_extremo
    
    X_train, X_test = train_test_split(df, test_size=TEST_SIZE, random_state=RANDOM_SEED)

    threshold_energia = X_train["wave_energy"].quantile(EXTREME_THRESHOLD)
    threshold_potencia = X_train["wave_power_kW_m"].quantile(EXTREME_THRESHOLD)

    scaler = StandardScaler()
    scaler.fit(
        X_train.loc[~es_extremo(X_train), MODEL_FEATURES]
    )

    X_train_no_extremo, X_train_extremo = procesar(X_train)
    X_test_no_extremo, X_test_extremo = procesar(X_test)

    gmm = GaussianMixture(
        n_components=N_CLUSTERS,
        covariance_type="full",
        random_state=RANDOM_SEED
    )

    gmm.fit(X_train_no_extremo[SCALED_MODEL_FEATURES])

    X_train_no_extremo["gmm_cluster"] = gmm.predict(X_train_no_extremo[SCALED_MODEL_FEATURES])
    X_train_no_extremo["gmm_cluster_probability"] = gmm.predict_proba(X_train_no_extremo[SCALED_MODEL_FEATURES]).max(axis=1)

    cluster_summary = (
        X_train_no_extremo.groupby("gmm_cluster")[EXTREME_FEATURES]
        .mean()
        .sort_values(EXTREME_FEATURES)
    )
    cluster_order = {
        old_cluster: f'{new_cluster + 1}'
        for new_cluster, old_cluster in enumerate(cluster_summary.index)
    }

    X_test_no_extremo["gmm_cluster"] = gmm.predict(X_test_no_extremo[SCALED_MODEL_FEATURES])
    X_test_no_extremo["gmm_cluster_probability"] = gmm.predict_proba(X_test_no_extremo[SCALED_MODEL_FEATURES]).max(axis=1)

    q_5 = X_test_no_extremo.groupby('gmm_cluster')['gmm_cluster_probability'].quantile(0.05)
    q_10 = X_test_no_extremo.groupby('gmm_cluster')['gmm_cluster_probability'].quantile(0.10)
    q_15 = X_test_no_extremo.groupby('gmm_cluster')['gmm_cluster_probability'].quantile(0.15)
    if not (
        all(X_test_no_extremo.groupby('gmm_cluster')['gmm_cluster_probability'].quantile(0.05)>=0.7) and
        all(X_test_no_extremo.groupby('gmm_cluster')['gmm_cluster_probability'].quantile(0.10)>=0.8) and
        all(X_test_no_extremo.groupby('gmm_cluster')['gmm_cluster_probability'].quantile(0.15)>=0.9) 
    ):
        print(f'El modelo GMM no cumple con los criterios')
        return None, None
    else:
        print(f'El modelo GMM cumple con los criterios')

    print(f'Quantiles: 0.05: {q_5}, 0.10: {q_10}, 0.15: {q_15}')

    X_test_no_extremo = X_test_no_extremo.assign(gmm_cluster=lambda df: df["gmm_cluster"].map(cluster_order))
    X_test_extremo = X_test_extremo.assign(gmm_cluster='7')

    X_train_no_extremo = X_train_no_extremo.assign(gmm_cluster=lambda df: df["gmm_cluster"].map(cluster_order))
    X_train_extremo = X_train_extremo.assign(gmm_cluster='7')

    df_train = pd.concat(
        [
            X_train_no_extremo[['coast_name', 'datetime'] + MODEL_FEATURES + ['gmm_cluster']],
            X_train_extremo[['coast_name', 'datetime'] + MODEL_FEATURES + ['gmm_cluster']]
        ],
        ignore_index=True
    )

    df_test = pd.concat(
        [
            X_test_no_extremo[['coast_name', 'datetime'] + MODEL_FEATURES + ['gmm_cluster']],
            X_test_extremo[['coast_name', 'datetime'] + MODEL_FEATURES + ['gmm_cluster']]
        ],
        ignore_index=True
    )

    return df_train, df_test


In [2]:
data = (
        spark.sql(
            f"""
                SELECT coast_name, datetime, {', '.join(set(MODEL_FEATURES + EXTREME_FEATURES))},
                CONCAT(coast_name, '_', DATE_FORMAT(datetime, 'yyyyMM')) AS coast_year_month
                FROM cor_{ambiente}.silver.swell_metrics
                WHERE YEAR(datetime) % {EVERY_N_YEARS} = 0
            """
        )
    )
coast_year_month_dict = {row.coast_year_month: PORCENTAJE_SAMPLE_DATA_MONTH for row in data.select('coast_year_month').distinct().collect()}
df = (
        data
        .sampleBy('coast_year_month', fractions=coast_year_month_dict, seed=RANDOM_SEED)
        .drop('coast_year_month')
    ).toPandas()

In [30]:
df_train, df_test = clasificar_gmm(df)

El modelo GMM cumple con los criterios
Quantiles: 0.05: gmm_cluster
0    0.975599
1    0.858306
2    0.864454
3    0.854856
4    0.959891
5    0.850566
Name: gmm_cluster_probability, dtype: float64, 0.10: gmm_cluster
0    0.999443
1    0.972031
2    0.975626
3    0.973996
4    0.999211
5    0.972637
Name: gmm_cluster_probability, dtype: float64, 0.15: gmm_cluster
0    0.999992
1    0.995292
2    0.996105
3    0.994925
4    0.999992
5    0.996124
Name: gmm_cluster_probability, dtype: float64


In [31]:
df_test.groupby('coast_name')['gmm_cluster'].value_counts(normalize=True).unstack(fill_value=0)

gmm_cluster,1,2,3,4,5,6,7
coast_name,,,,,,,
Matamoros,0.238370,0.357188,0.162380,0.101739,0.062788,0.050267,0.027268
Playa del Carmen,0.821153,0.123750,0.023908,0.014388,0.009081,0.005045,0.002676
Puerto Vallarta,0.000000,0.093274,0.411750,0.307050,0.135900,0.037075,0.014951
Punta Colonet,0.000000,0.067597,0.259150,0.276017,0.195635,0.098127,0.103474
Sabancuy,0.693170,0.224200,0.041930,0.018460,0.011032,0.007516,0.003692
Salina Cruz,0.004132,0.246908,0.261425,0.174809,0.137292,0.122992,0.052443
Tuxpan,0.434544,0.356792,0.095211,0.040917,0.030863,0.025890,0.015783


In [32]:
df_train, df_test = clasificar_gmm(df)

El modelo GMM cumple con los criterios
Quantiles: 0.05: gmm_cluster
0    0.972510
1    0.854698
2    0.869056
3    0.856100
4    0.957552
5    0.862092
Name: gmm_cluster_probability, dtype: float64, 0.10: gmm_cluster
0    0.999253
1    0.970588
2    0.977395
3    0.969971
4    0.999166
5    0.976715
Name: gmm_cluster_probability, dtype: float64, 0.15: gmm_cluster
0    0.999989
1    0.995100
2    0.996443
3    0.993647
4    0.999996
5    0.996553
Name: gmm_cluster_probability, dtype: float64


In [33]:
df_test.groupby('coast_name')['gmm_cluster'].value_counts(normalize=True).unstack(fill_value=0)

gmm_cluster,1,2,3,4,5,6,7
coast_name,,,,,,,
Matamoros,0.216576,0.348465,0.173826,0.107764,0.069153,0.056947,0.027268
Playa del Carmen,0.804834,0.134848,0.027022,0.014345,0.010309,0.005966,0.002676
Puerto Vallarta,0.000000,0.064587,0.377836,0.327269,0.157091,0.058266,0.014951
Punta Colonet,0.000000,0.046855,0.234276,0.276327,0.208601,0.130466,0.103474
Sabancuy,0.666799,0.238968,0.049270,0.020218,0.012131,0.008922,0.003692
Salina Cruz,0.002936,0.211348,0.265857,0.179268,0.148193,0.139956,0.052443
Tuxpan,0.405167,0.364953,0.107048,0.044511,0.033241,0.029296,0.015783


In [34]:
df_train, df_test = clasificar_gmm(df)

El modelo GMM cumple con los criterios
Quantiles: 0.05: gmm_cluster
0    0.972510
1    0.854698
2    0.869056
3    0.856100
4    0.957552
5    0.862092
Name: gmm_cluster_probability, dtype: float64, 0.10: gmm_cluster
0    0.999253
1    0.970588
2    0.977395
3    0.969971
4    0.999166
5    0.976715
Name: gmm_cluster_probability, dtype: float64, 0.15: gmm_cluster
0    0.999989
1    0.995100
2    0.996443
3    0.993647
4    0.999996
5    0.996553
Name: gmm_cluster_probability, dtype: float64


In [35]:
df_test.groupby('coast_name')['gmm_cluster'].value_counts(normalize=True).unstack(fill_value=0)

gmm_cluster,1,2,3,4,5,6,7
coast_name,,,,,,,
Matamoros,0.216576,0.348465,0.173826,0.107764,0.069153,0.056947,0.027268
Playa del Carmen,0.804834,0.134848,0.027022,0.014345,0.010309,0.005966,0.002676
Puerto Vallarta,0.000000,0.064587,0.377836,0.327269,0.157091,0.058266,0.014951
Punta Colonet,0.000000,0.046855,0.234276,0.276327,0.208601,0.130466,0.103474
Sabancuy,0.666799,0.238968,0.049270,0.020218,0.012131,0.008922,0.003692
Salina Cruz,0.002936,0.211348,0.265857,0.179268,0.148193,0.139956,0.052443
Tuxpan,0.405167,0.364953,0.107048,0.044511,0.033241,0.029296,0.015783
